# 第 5b 章：PyTorch 速成——从 numpy 到 `loss.backward()`

> **写给谁**：Phase 1 我们全程用 numpy（把认知负载留给 RL 本身）；从 Ch06 的 DQN 开始要用 PyTorch。
> 如果你**写过 PyTorch / 熟悉 autograd**，直接跳到本章末尾的"5b.6 揭开 Ch06 的黑盒"即可。
> 如果你只用过 numpy——这一章就是为你写的，花 1 小时，后面 13 章都会顺很多。

## 学习目标

读完本章后你应该能：

1. 用 `torch.Tensor` 做你熟悉的一切 numpy 操作
2. 说清楚 `requires_grad` / `backward()` / `.grad` 三者的关系
3. 默写**标准训练五步循环**：`zero_grad → forward → loss → backward → step`
4. 定义一个 `nn.Module`，并解释 `train()` / `eval()` 的区别
5. 看懂 Ch06 会用到的三个操作：`gather`、`torch.no_grad`、target network 拷贝

## 为什么从 Ch06 起切换 PyTorch？

Ch05 的 Q-learning 用**表格**存 $Q(s,a)$——状态太多时表格爆炸（CartPole 的连续状态就存不下）。
Ch06 起我们用**神经网络**逼近 $Q$，而"网络参数 $\theta$ 关于 loss 的梯度"
交给 PyTorch 的 autograd 自动算：**你写前向，它算反向**。我们手写 replay buffer、训练循环、
PPO clip——只有求导这一步交给框架（这是本教材"不依赖黑盒 RL 库"原则的边界）。

In [ ]:
# 自动设置 sys.path（和 Ch00 一样）
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import torch
except ImportError:
    raise SystemExit(
        "本章需要 PyTorch：请先 pip install torch（CPU 版即可），再重启 kernel 重跑。"
    )

import numpy as np
print(f"torch: {torch.__version__}  |  numpy: {np.__version__}")

## 5b.1 Tensor：会 numpy 就会一半

`torch.Tensor` 和 `np.ndarray` 的 API 高度一致——下面这张对照表覆盖了本教材用到的全部操作：

| 你在 numpy 里写的 | PyTorch 里写 | 备注 |
|---|---|---|
| `np.zeros((3,4))` | `torch.zeros(3,4)` | shape 不再是 tuple |
| `np.ones(5)` | `torch.ones(5)` | |
| `A @ B` | `A @ B` | 矩阵乘完全一样 |
| `A.sum(axis=1)` | `A.sum(dim=1)` | `axis` 改叫 `dim` |
| `np.argmax(A)` | `A.argmax()` | |
| `A[0:2]` | `A[0:2]` | 切片一样 |
| `A[[0,2]]` | `A[[0,2]]` 或 `A.index_select(0, idx)` | |
| `rng.uniform(...)` | `torch.rand(...)` | 随机数见下 |

In [ ]:
import torch

# 创建 + 运算：和 numpy 几乎逐字对应
A = torch.rand(2, 3)            # 均匀分布 [0,1)，相当于 rng.uniform(0,1,(2,3))
B = torch.randn(2, 3)           # 标准正态
print("A =", A)
print("A.sum(dim=1) =", A.sum(dim=1))
print("A.argmax(dim=1) =", A.argmax(dim=1))

# 和 numpy 互转（共享内存，零拷贝——改一边另一边也变！）
a_np = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(a_np)      # numpy -> torch
back = t.numpy()                # torch -> numpy
print("torch.from_numpy:", t, "| .numpy():", back)

# 默认 dtype 是 float32（不是 numpy 的 float64）——神经网络的标准精度
print("默认 dtype:", torch.ones(1).dtype)

**两个和 numpy 不同的习惯**要提前记住：

1. **随机性**：`torch.manual_seed(0)` 对应 `np.random.seed(0)`；本教材统一用 `from utils import set_seed`，它会同时 seed numpy / random / torch。
2. **广播（broadcasting）规则与 numpy 完全一致**：`[B, T, V] * [B, T, 1]` 会自动广播到最后一个维度。后面章节大量依赖这个。

## 5b.2 autograd：PyTorch 的灵魂

一句话：**你用 tensor 算出 loss，调 `loss.backward()`，PyTorch 自动算出 loss 关于所有"标记了 `requires_grad=True` 的 tensor"的梯度**，存进它们的 `.grad` 属性。

$$
\text{设 } f(x) = 3x^2 + 2x \quad \Rightarrow \quad \frac{df}{dx} = 6x + 2
$$

In [ ]:
x = torch.tensor([1.0], requires_grad=True)   # 标记：我要对 x 求导
f = 3 * x**2 + 2 * x
f.backward()                                    # 反向传播：自动算 df/dx
print(f"f(x=1) = {f.item()}")
print(f"解析梯度 df/dx = 6*1+2 = 8.0")
print(f"autograd 算出 x.grad = {x.grad.item()}")

# 教材传统：数值验证——用有限差分 (f(x+h) - f(x-h)) / (2h) 对比
# 注意用 float64：float32 下 x±h 的舍入误差被 /2h 放大，差分会不准
xd = torch.tensor([1.0], dtype=torch.float64)
h = 1e-6
fd = ((3*(xd+h)**2 + 2*(xd+h)) - (3*(xd-h)**2 + 2*(xd-h))) / (2*h)
print(f"有限差分梯度          = {fd.item():.8f}")
print(f"解析 / autograd       = 8.0 / {x.grad.item()}")
print("两者一致 ✓" if abs(fd.item() - x.grad.item()) < 1e-4 else "不一致 ✗")

注意上面第一次出现的 `with torch.no_grad():`——**在块内的运算不记录梯度**。
为什么需要它：算"验证用的数值"或"target 值"时我们不需要梯度，关掉能省内存、避免误更新。

### 训练五步循环（背下来）

```python
optimizer.zero_grad()   # ① 清上一次的梯度（梯度默认累加，不清零会越加越大）
loss = criterion(model(x), y)   # ② 前向：算预测 + loss
loss.backward()         # ③ 反向：算 d loss / d 所有参数
optimizer.step()        # ④ 用梯度更新参数（如 θ -= lr * grad）
# ⑤ 重复 ①-④，直到 loss 收敛
```

> 最常见的初学者 bug：**忘写 ①**。PyTorch 的梯度是累加的（`.grad +=`），不清零等于用了一个越来越大的错误梯度。

In [ ]:
# 完整最小示例：用神经网络拟合 y = sin(x)（CPU 上几秒）
from utils import set_seed
set_seed(0)

# 待拟合的数据
x = torch.linspace(-3, 3, 200).unsqueeze(1)    # [200, 1]
y = torch.sin(x)

# 一个两层 MLP（5b.3 会详细讲 nn.Module）
model = torch.nn.Sequential(
    torch.nn.Linear(1, 64), torch.nn.ReLU(),
    torch.nn.Linear(64, 1),
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)   # Adam：自适应步长
criterion = torch.nn.MSELoss()

losses = []
for step in range(500):
    optimizer.zero_grad()          # ①
    pred = model(x)                # ② 前向
    loss = criterion(pred, y)
    loss.backward()                # ③
    optimizer.step()               # ④
    losses.append(loss.item())     # ⑤

print(f"loss: {losses[0]:.4f} -> {losses[-1]:.4f}")

In [ ]:
# 看看拟合效果
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(losses)
axes[0].set_title("training loss")
axes[0].set_yscale("log")
with torch.no_grad():               # 画图不需要梯度
    axes[1].scatter(x, y, s=6, label="data: sin(x)")
    axes[1].plot(x, model(x), color="red", label="network")
axes[1].legend()
axes[1].set_title("fit result")
plt.show()

## 5b.3 `nn.Module`：所有网络的基类

后面章节的 Q 网络、策略网络、TinyGPT 全是 `nn.Module` 的子类。它替你做三件事：

1. **登记参数**：`self.linear = nn.Linear(...)` 里的权重会自动进 `model.parameters()`，optimizer 拿这个列表去更新
2. **`model(x)` 自动调 `forward`**：`__call__` 会挂钩子（hook），别直接调 `model.forward(x)`
3. **模式切换**：`model.train()` / `model.eval()` 控制 dropout、batchnorm 等层的行为

> ⚠️ **第三点是本教材代码里真实踩过的坑**：eval() 之后忘了切回 train()，后续训练就在错误的模式下跑。
> 记住配对规则——**rollout / 采样 / 评估时 eval()，用完立刻 train() 回来**。

In [ ]:
class TinyMLP(torch.nn.Module):
    def __init__(self, in_dim: int, hidden: int, out_dim: int):
        super().__init__()                          # 必须先调父类构造
        self.net = torch.nn.Sequential(             # 子模块也会被登记
            torch.nn.Linear(in_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, out_dim),
        )

    def forward(self, x):                           # 只定义前向
        return self.net(x)

m = TinyMLP(4, 32, 2)
print(m)                                            # 自动生成的结构描述
print(f"参数量: {sum(p.numel() for p in m.parameters())}")   # (4*32+32) + (32*2+2) = 226

# train/eval 模式查看
print("默认 training 模式:", m.training)   # True
m.eval()
print("eval() 后:", m.training)            # False
m.train()
print("train() 后:", m.training)           # True

## 5b.4 Ch06 特供：三个马上要用的操作

**① `gather`——按索引取 Q 值**。DQN 的 loss 需要 $Q(s_t, a_t)$：网络输出**所有**动作的 Q `[B, n_actions]`，但我们只要**实际执行的那个动作**的 Q。这就是 gather：

```python
q = q_net(states)                      # [B, n_actions]
q_sa = q.gather(1, actions)            # [B, 1]  取第 b 行第 actions[b] 个
```

**② `torch.no_grad()`——构造 TD target**。target $= r + \gamma \max_{a'} Q_{\text{target}}(s', a')$ 是**监督信号**，不该有梯度流过（否则会"追自己的尾巴"，即 Ch06 §6.x 的半梯度概念）。

**③ target network 拷贝**：`target_net.load_state_dict(online_net.state_dict())` 把在线网络整套参数复制给目标网络（隔 N 步做一次，稳定训练）。

In [ ]:
from utils.networks import QNetwork   # Ch06 用的真实 Q 网络（5b.6 揭晓内部）

set_seed(0)
q_net = QNetwork(state_dim=4, n_actions=2)     # CartPoleLite 的形状

states  = torch.rand(8, 4)                     # batch = 8 条 transition
actions = torch.randint(0, 2, (8, 1))          # 每条实际执行的动作 [B, 1]

# ① gather
q_all = q_net(states)                          # [8, 2]
q_sa = q_all.gather(1, actions)                # [8, 1]
print("q_all[0] =", q_all[0].tolist(), "| action[0] =", actions[0].item(),
      "| gather 对得上:", abs(q_all[0, actions[0].item()].item() - q_sa[0].item()) < 1e-7)

# ② no_grad 构造 target（模仿 dqn_utils.dqn_update_step 里的关键两行）
rewards, dones = torch.ones(8), torch.zeros(8)
with torch.no_grad():
    q_next_max = q_net(states).max(dim=1).values       # 假装这是 target_net
    target = rewards + 0.99 * q_next_max * (1 - dones)
print("target 无梯度（detach 于 no_grad）:", not target.requires_grad)

# ③ target network 拷贝
target_net = QNetwork(state_dim=4, n_actions=2)       # 初始随机（和 q_net 不同）
target_net.load_state_dict(q_net.state_dict())        # 现在完全一致
with torch.no_grad():
    diff = (q_net(states) - target_net(states)).abs().max()
print(f"拷贝后两网络输出最大差异: {diff.item():.2e}  (应为 0)")

## 5b.5 揭开 Ch06 的黑盒：`utils/networks.py` 的 `QNetwork`

Ch06 直接 `from utils.networks import QNetwork`——现在你已经能完全读懂它了。逐行对照上面的知识：

- `make_mlp(...)`：把 `[Linear, ReLU, Linear, ...]` 叠成 `nn.Sequential`（就是 5b.3 的 TinyMLP 换个写法）
- `forward`：`x -> self.net(x)`，输入 state `[B, 4]`，输出 Q 值 `[B, n_actions]`

In [ ]:
import inspect
from utils.networks import make_mlp, QNetwork
print(inspect.getsource(QNetwork))
# 练习（不用写代码）：说出 QNetwork(state_dim=4, n_actions=2) 有多少参数、
# forward 的输入输出 shape——答不上来就回 5b.3 / 5b.4 再看一遍。

## 5b.6 常见坑速查（后面章节踩到时回来翻）

| 症状 | 原因 | 修复 |
|---|---|---|
| loss 不降反升 / 梯度爆炸 | 忘了 `optimizer.zero_grad()` | 五步循环①别省 |
| 显存越跑越满 | 存了带梯度的张量（如整个 loss 历史） | 用 `.item()` / `.detach()` 只存数值 |
| `RuntimeError: element 0 of tensors does not require grad` | 在 `no_grad` 块里做训练 | 检查缩进，训练代码别放进 no_grad |
| 评估指标随机跳变 | dropout 还开着（没 eval()） | 评估前 `model.eval()`，用完 `model.train()` |
| numpy 和 tensor 混用报错 | `np.ndarray` 直接喂网络 | `torch.as_tensor(x, dtype=torch.float32)` |

## 小结

- ✅ tensor ≈ numpy（`axis`→`dim`，默认 float32）
- ✅ autograd：`requires_grad` 标记 → `backward()` 计算 → `.grad` 读取；数值验证用有限差分
- ✅ 训练五步循环：`zero_grad → forward → loss → backward → step`
- ✅ `nn.Module` 管参数 / train / eval 模式
- ✅ DQN 三件套：`gather` 取 $Q(s,a)$、`no_grad` 造 target、`load_state_dict` 拷贝网络

下一章：**第 6 章 — DQN + 函数逼近**。神经网络进场，replace 表格——但探索、replay、target network 这些 RL 的核心难题，全部手写。

> 📖 学完 Ch06 后记得做 `STUDY_GUIDE.md` 里 Ch06 的自测题。